# Task 03 — DLRM from Scratch
**Model:** Deep Learning Recommendation Model (Naumov et al., 2019)  
**Data:** Criteo-style CTR dataset in `../../dataset_2_3/`  
**Pipeline:** `common/ctr_data.py`, `common/metrics.py`, `common/utils.py`  
**Logic:** `03_DLRM/dlrm.py`, `02_NeuralCTR/train.py` (reused)

## How to rerun
```bash
pip install -r RecommendationTasks/requirements.txt
# Dataset should be at: Mandatory_Tasks/Recommender_Systems/dataset_2_3/
jupyter nbconvert --to notebook --execute dlrm.ipynb
```

In [ ]:
# Cell 1 — Environment check
import sys, platform
import torch
print(f"Python  : {platform.python_version()}")
print(f"PyTorch : {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device  : {device}" + (f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))
print(f"Seed    : 42")

In [ ]:
# Cell 2 — Imports and paths
import pathlib, json, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# Add RecommendationTasks/ to path so common/ is importable
ROOT = pathlib.Path('..').resolve()           # RecommendationTasks/
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / '02_NeuralCTR'))  # reuse train.py

from common.utils import set_seed, get_device, count_params
from common.ctr_data import load_ctr_data
from common.metrics import compute_metrics, find_best_f1_threshold, plot_reliability_diagram
from train import train_model, eval_on_test, save_results
from dlrm import DLRM

set_seed(42)
DEVICE = get_device()

DATA_DIR    = pathlib.Path('../../dataset_2_3').resolve()
SPLIT_PATH  = ROOT / 'common' / 'split.npz'    # same split as Task 02
RESULTS_DIR = pathlib.Path('results')
FIGURES_DIR = pathlib.Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Data dir : {DATA_DIR}")
print(f"Split    : {SPLIT_PATH} (shared with Task 02)")

In [ ]:
# Cell 3 — Load data (reuses the exact same 90/10 split.npz as Task 02)
train_loader, val_loader, test_loader, NUM_NUMERIC, CAT_CARDS = load_ctr_data(
    data_dir=DATA_DIR,
    split_save_path=SPLIT_PATH,
    batch_size=2048,
    min_freq=10,
    seed=42,
)

print(f"Numeric features  : {NUM_NUMERIC}")
print(f"Categorical fields: {len(CAT_CARDS)}")
print(f"Cardinalities (first 5): {CAT_CARDS[:5]}  ...  (last 5): {CAT_CARDS[-5:]}")
print(f"Train batches : {len(train_loader)} (batch=2048)")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")

In [ ]:
# Cell 4 — DLRM shape sanity check (run before full training)
import torch
set_seed(42)

_model_test = DLRM(
    num_numeric=NUM_NUMERIC,
    cat_cardinalities=CAT_CARDS,
    embed_dim=16,
    bottom_mlp_sizes=[512, 256],
    top_mlp_sizes=[512, 256],
)
_x_num = torch.randn(4, NUM_NUMERIC)
_x_cat = torch.randint(0, 5, (4, len(CAT_CARDS)))
_logits = _model_test(_x_num, _x_cat)

print(f"n_cat            : {_model_test.n_cat}   (26 fields)")
print(f"n_interactions   : {_model_test.n_interactions}  (27*26/2)")
print(f"top MLP input dim: 16 + {_model_test.n_interactions} = {16 + _model_test.n_interactions}")
print(f"logits shape     : {list(_logits.shape)}  ✓" if _logits.shape == (4,) else f"SHAPE MISMATCH: {_logits.shape}")
print(f"Total params     : {count_params(_model_test):,}")
del _model_test, _x_num, _x_cat, _logits

In [ ]:
# Cell 5 — Helper: run one DLRM experiment and save results
def run_dlrm_experiment(name, model, tag=None):
    tag = tag or name
    print(f"\n{'='*60}")
    print(f"  Training: {name}  ({count_params(model):,} params)")
    print(f"{'='*60}")

    set_seed(42)  # reset per run for fair comparison
    run_info = train_model(
        model, train_loader, val_loader,
        lr=1e-3, weight_decay=1e-5,
        max_epochs=10, patience=2,
        device=DEVICE,
    )

    result = {
        "model": name,
        "n_params": run_info["n_params"],
        "best_val_loss": run_info["best_val_loss"],
        "best_epoch": run_info["best_epoch"],
        "history": run_info["history"],
        "peak_memory_mb": run_info["peak_memory_mb"],
        "best_threshold": run_info["best_threshold"],
    }
    save_results(result, RESULTS_DIR / f"{tag}.json")
    return model, run_info

all_results = {}   # name -> (model, run_info)

In [ ]:
# Cell 6 — Train DLRM (d=16, standard configuration)
# Architecture: Bottom MLP [13→512→256→16], 26 Embeddings (dim=16),
# Interaction (351 pairwise dots), Top MLP [367→512→256→1]
dlrm_d16 = DLRM(
    num_numeric=NUM_NUMERIC,
    cat_cardinalities=CAT_CARDS,
    embed_dim=16,
    bottom_mlp_sizes=[512, 256],
    top_mlp_sizes=[512, 256],
)
all_results['DLRM-d16'] = run_dlrm_experiment('DLRM (d=16)', dlrm_d16, tag='dlrm_d16')

In [ ]:
# Cell 7 — Ablation 1: No-interaction (concat) baseline
# Instead of the dot-product interaction block, we directly concatenate the
# bottom MLP output and all 26 embeddings, then pass the concat into a top MLP.
# This tests whether explicit second-order dot-product interactions help.

import torch.nn as nn

class DLRMNoInteraction(nn.Module):
    """DLRM variant where the interaction layer is replaced by a plain concat.
    
    Input to top MLP: [dense_out | emb_0 | emb_1 | ... | emb_25]
    Size: d + 26*d = 27*d  (vs d + 351 in the full DLRM)
    """
    def __init__(self, num_numeric, cat_cardinalities, embed_dim=16,
                 bottom_mlp_sizes=None, top_mlp_sizes=None, top_dropout=0.0):
        super().__init__()
        if bottom_mlp_sizes is None:
            bottom_mlp_sizes = [512, 256]
        if top_mlp_sizes is None:
            top_mlp_sizes = [512, 256]

        self.embed_dim = embed_dim
        self.n_cat = len(cat_cardinalities)

        # Bottom MLP
        from dlrm import _make_mlp
        bottom_sizes = [num_numeric] + bottom_mlp_sizes + [embed_dim]
        self.bottom_mlp = _make_mlp(bottom_sizes)

        # Embeddings
        self.embeddings = nn.ModuleList([
            nn.Embedding(card, embed_dim) for card in cat_cardinalities
        ])
        bound = 1.0 / (embed_dim ** 0.5)
        for emb in self.embeddings:
            nn.init.uniform_(emb.weight, -bound, bound)

        # Top MLP: concat of all 27 vectors → 27*d
        top_input_dim = (self.n_cat + 1) * embed_dim  # 27 * d
        top_sizes = [top_input_dim] + top_mlp_sizes + [1]
        self.top_mlp = _make_mlp(top_sizes, dropout=top_dropout)

    def forward(self, x_num, x_cat):
        dense_out = self.bottom_mlp(x_num)   # [B, d]
        emb_outs = [emb(x_cat[:, j]) for j, emb in enumerate(self.embeddings)]  # 26 x [B, d]
        # Concatenate everything: [B, 27*d]
        concat = torch.cat([dense_out] + emb_outs, dim=1)
        logit = self.top_mlp(concat)          # [B, 1]
        return logit.squeeze(1)               # [B]

dlrm_no_interact = DLRMNoInteraction(
    num_numeric=NUM_NUMERIC,
    cat_cardinalities=CAT_CARDS,
    embed_dim=16,
    bottom_mlp_sizes=[512, 256],
    top_mlp_sizes=[512, 256],
)
print(f"No-interaction top-MLP input dim: {(len(CAT_CARDS)+1)*16} (vs {16 + 351} for full DLRM)")
all_results['DLRM-no-interact'] = run_dlrm_experiment('DLRM (no-interaction)', dlrm_no_interact, tag='dlrm_no_interact')

In [ ]:
# Cell 8 — Ablation 2a: Embedding dim d=8
dlrm_d8 = DLRM(
    num_numeric=NUM_NUMERIC,
    cat_cardinalities=CAT_CARDS,
    embed_dim=8,
    bottom_mlp_sizes=[512, 256],
    top_mlp_sizes=[512, 256],
)
# With d=8: n_interactions = 27*26/2 = 351, top input = 8+351 = 359
all_results['DLRM-d8'] = run_dlrm_experiment('DLRM (d=8)', dlrm_d8, tag='dlrm_d8')

In [ ]:
# Cell 9 — Ablation 2b: Embedding dim d=32
dlrm_d32 = DLRM(
    num_numeric=NUM_NUMERIC,
    cat_cardinalities=CAT_CARDS,
    embed_dim=32,
    bottom_mlp_sizes=[512, 256],
    top_mlp_sizes=[512, 256],
)
# With d=32: top input = 32+351 = 383
all_results['DLRM-d32'] = run_dlrm_experiment('DLRM (d=32)', dlrm_d32, tag='dlrm_d32')

In [ ]:
# Cell 10 — Pick best DLRM by val log loss and evaluate ONCE on test
best_name = min(
    all_results,
    key=lambda n: all_results[n][1]['best_val_loss']
)
best_model, best_info = all_results[best_name]
best_threshold = best_info['best_threshold']

print(f"Best DLRM by val log loss: {best_name}")
print(f"  best_val_loss = {best_info['best_val_loss']:.4f}  at epoch {best_info['best_epoch']}")
print(f"  F1-optimal threshold (from val): {best_threshold:.3f}")

test_metrics = eval_on_test(best_model, test_loader, threshold=best_threshold, device=DEVICE)
print("\nTest metrics:")
for k, v in test_metrics.items():
    print(f"  {k}: {v}")
save_results(
    {"model": best_name, "threshold": best_threshold, **test_metrics},
    RESULTS_DIR / 'dlrm_best_test.json'
)

In [ ]:
# Cell 11 — Full metrics table for all DLRM variants (val set)
import pandas as pd
import torch.nn as nn
from train import evaluate

criterion = nn.BCEWithLogitsLoss()
rows = []

for name, (model, info) in all_results.items():
    _, y_true_val, y_prob_val = evaluate(model, val_loader, criterion, DEVICE)
    thresh = info['best_threshold'] or find_best_f1_threshold(y_true_val, y_prob_val)
    m = compute_metrics(y_true_val, y_prob_val, threshold=thresh)

    rows.append({
        'Model': name,
        'ROC-AUC': round(m['roc_auc'], 4),
        'PR-AUC': round(m['pr_auc'], 4),
        'Log Loss': round(m['log_loss'], 4),
        'Accuracy': round(m['accuracy_0.5'], 4),
        'F1': round(m['f1'], 4),
        'Precision': round(m['precision'], 4),
        'Recall': round(m['recall'], 4),
        'ECE': round(m['ece'], 4),
        'Threshold': round(thresh, 3),
        'Params': info['n_params'],
        'Peak Mem (MB)': info['peak_memory_mb'],
        'Sec/Epoch': round(sum(info['history']['epoch_seconds']) / len(info['history']['epoch_seconds']), 1),
        'Best Epoch': info['best_epoch'],
    })

df = pd.DataFrame(rows).sort_values('Log Loss')
print(df.to_string(index=False))
df.to_csv(RESULTS_DIR / 'dlrm_summary_table.csv', index=False)
print('\nSaved → results/dlrm_summary_table.csv')

In [ ]:
# Cell 12 — Loss curves for all DLRM variants
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=False)
axes = axes.flatten()

for ax, (name, (model, info)) in zip(axes, all_results.items()):
    hist = info['history']
    epochs = range(1, len(hist['train_loss']) + 1)
    ax.plot(epochs, hist['train_loss'], 'o-', label='Train loss', color='steelblue')
    ax.plot(epochs, hist['val_loss'], 's--', label='Val loss', color='crimson')
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('BCE Loss')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.4)

# Hide any unused subplot
for ax in axes[len(all_results):]:
    ax.set_visible(False)

plt.suptitle('DLRM — Train vs Val Loss Curves (All Variants)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dlrm_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → figures/dlrm_loss_curves.png')

In [ ]:
# Cell 13 — Embedding dim ablation bar chart (ROC-AUC vs d)
dim_models = ['DLRM-d8', 'DLRM-d16', 'DLRM-d32']
dims   = [8, 16, 32]
aucs   = []
losses = []

for tag in dim_models:
    if tag in all_results:
        _, info = all_results[tag]
        _, y_t, y_p = evaluate(all_results[tag][0], val_loader, nn.BCEWithLogitsLoss(), DEVICE)
        m = compute_metrics(y_t, y_p)
        aucs.append(m['roc_auc'])
        losses.append(m['log_loss'])
    else:
        aucs.append(None)
        losses.append(None)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.bar([str(d) for d in dims], aucs, color=['#4e79a7', '#f28e2b', '#59a14f'])
ax1.set_xlabel('Embedding Dimension d')
ax1.set_ylabel('ROC-AUC (val)')
ax1.set_title('DLRM: ROC-AUC vs Embedding Dim')
ax1.grid(True, axis='y', linestyle='--', alpha=0.4)
for i, v in enumerate(aucs):
    if v:
        ax1.text(i, v + 0.001, f"{v:.4f}", ha='center', fontsize=9)

ax2.bar([str(d) for d in dims], losses, color=['#4e79a7', '#f28e2b', '#59a14f'])
ax2.set_xlabel('Embedding Dimension d')
ax2.set_ylabel('Log Loss (val)')
ax2.set_title('DLRM: Log Loss vs Embedding Dim')
ax2.grid(True, axis='y', linestyle='--', alpha=0.4)
for i, v in enumerate(losses):
    if v:
        ax2.text(i, v + 0.0005, f"{v:.4f}", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dlrm_embed_dim_ablation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → figures/dlrm_embed_dim_ablation.png')

In [ ]:
# Cell 14 — Interaction ablation bar chart: DLRM-d16 vs no-interaction
ablation_names = ['DLRM-d16', 'DLRM-no-interact']
ab_aucs   = []
ab_losses = []
ab_labels = []

for tag in ablation_names:
    if tag in all_results:
        model, info = all_results[tag]
        _, y_t, y_p = evaluate(model, val_loader, nn.BCEWithLogitsLoss(), DEVICE)
        m = compute_metrics(y_t, y_p)
        ab_aucs.append(m['roc_auc'])
        ab_losses.append(m['log_loss'])
        ab_labels.append('DLRM\n(dot-product)' if 'no' not in tag else 'DLRM\n(no-interaction)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))

colors = ['#4e79a7', '#e15759']
ax1.bar(ab_labels, ab_aucs, color=colors)
ax1.set_ylabel('ROC-AUC (val)')
ax1.set_title('Interaction Ablation: ROC-AUC')
ax1.grid(True, axis='y', linestyle='--', alpha=0.4)
for i, v in enumerate(ab_aucs):
    ax1.text(i, v + 0.001, f"{v:.4f}", ha='center', fontsize=9)

ax2.bar(ab_labels, ab_losses, color=colors)
ax2.set_ylabel('Log Loss (val)')
ax2.set_title('Interaction Ablation: Log Loss')
ax2.grid(True, axis='y', linestyle='--', alpha=0.4)
for i, v in enumerate(ab_losses):
    ax2.text(i, v + 0.0005, f"{v:.4f}", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dlrm_interaction_ablation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → figures/dlrm_interaction_ablation.png')

In [ ]:
# Cell 15 — Reliability diagram for best DLRM (val set)
_, y_true_val, y_prob_val = evaluate(best_model, val_loader, nn.BCEWithLogitsLoss(), DEVICE)

plot_reliability_diagram(
    y_true_val, y_prob_val,
    model_name=best_name,
    save_path=FIGURES_DIR / 'dlrm_reliability_diagram.png'
)
print('Saved → figures/dlrm_reliability_diagram.png')

## Analysis

### DLRM vs MF (Task 01)
- **MF** factorizes a user–item matrix into two embedding tables and predicts a rating as a dot product. It only sees user/item IDs — no side features, no context.
- **DLRM** handles both dense numeric features (e.g., time-on-page) and sparse categorical features (e.g., ad category, publisher). It predicts a click probability for a *new impression*, not a rating for a known user-item pair.

### DLRM vs vanilla MLP (Task 02)
- A plain MLP receives the concatenated embeddings and has to *discover* pairwise interactions through depth. This requires at least two layers to model second-order terms.
- DLRM's dot-product interaction layer computes all 351 second-order terms *explicitly* in a single batched matrix multiply `X @ X^T`. The top MLP then only needs to weight pre-computed interactions rather than re-discover them.

### Costs
- **Memory:** Embedding tables dominate. Total embedding params ≈ sum(cardinalities) × d. At d=16 and real Criteo scale (cardinalities in the millions), this is gigabytes per model.
- **Complexity:** More hyperparameters (embed_dim, bottom MLP depth/width, top MLP depth/width) vs the vanilla MLP.
- **At production scale:** The embedding tables are sharded across many GPUs (model parallelism), while the bottom/top MLPs are replicated (data parallelism).

### Observed results
If the gain over the best MLP is modest on this subsample, likely reasons:
1. The training data is small relative to the number of embedding parameters → embeddings overfit quickly.
2. Short training (early stopping at ≤3 epochs) limits how much the interaction layer can be learned.
3. Our Criteo-style dataset is a fraction of the 45M-row dataset in the original paper.